<a href="https://colab.research.google.com/github/Tariik7/BIGDATALABS/blob/main/TP_SPARK_STREAMING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pyspark


In [19]:
# Serveur socket simulant un flux
import socket, time, threading
def start_socket_server():
 host = "localhost"
 port = 9999
 s = socket.socket()
 s.bind((host, port))
 s.listen(1)
 conn, addr = s.accept()
 messages = ["spark streaming dstream", "spark spark streaming", "big dataspark"]
 while True:
  for msg in messages:
       conn.send((msg + "\n").encode())
       time.sleep(2)
threading.Thread(target=start_socket_server, daemon=True).start()

from pyspark import SparkContext
from pyspark.streaming import StreamingContext
sc = SparkContext.getOrCreate()
ssc = StreamingContext(sc, 5)
lines = ssc.socketTextStream("localhost", 9999)
words = lines.flatMap(lambda line: line.split(" "))
pairs = words.map(lambda w: (w, 1))
counts = pairs.reduceByKey(lambda a, b: a + b)
counts.pprint()
ssc.start()
ssc.awaitTerminationOrTimeout(30)
ssc.stop(stopSparkContext=False)

-------------------------------------------
Time: 2026-01-16 20:06:15
-------------------------------------------
('streaming', 1)
('dstream', 1)
('spark', 1)

-------------------------------------------
Time: 2026-01-16 20:06:20
-------------------------------------------
('streaming', 2)
('big', 1)
('dstream', 1)
('spark', 3)
('dataspark', 1)

-------------------------------------------
Time: 2026-01-16 20:06:25
-------------------------------------------
('streaming', 1)
('big', 1)
('spark', 2)
('dataspark', 1)

-------------------------------------------
Time: 2026-01-16 20:06:30
-------------------------------------------
('streaming', 2)
('dstream', 1)
('big', 1)
('spark', 3)
('dataspark', 1)

-------------------------------------------
Time: 2026-01-16 20:06:35
-------------------------------------------
('streaming', 2)
('dstream', 1)
('spark', 3)

-------------------------------------------
Time: 2026-01-16 20:06:40
-------------------------------------------
('big', 1)
('stre

In [20]:

# Installation de la bibliothèque PySpark nécessaire pour le streaming
!pip install pyspark


# PARTIE 2 : CRÉATION DU SERVEUR SOCKET (SIMULATION DU FLUX DE DONNÉES)
# Ce serveur simule une source de données en temps réel en envoyant des messages
# via un socket TCP sur le port 9999

import socket
import time
import threading

def start_socket_server():
    """
    Fonction qui démarre un serveur socket pour simuler un flux de données.
    - Écoute sur localhost:9999
    - Envoie cycliquement des messages toutes les 2 secondes
    """
    host = "localhost"
    port = 9999

    # Création et configuration du socket
    s = socket.socket()
    s.bind((host, port))
    s.listen(1)

    print(f"[SERVEUR] En attente de connexion sur {host}:{port}...")
    conn, addr = s.accept()
    print(f"[SERVEUR] Client connecté depuis {addr}")

    # Messages à envoyer en boucle
    messages = [
        "spark streaming dstream",
        "spark spark streaming",
        "big data spark"
    ]

    # Envoi cyclique des messages
    while True:
        for msg in messages:
            conn.send((msg + "\n").encode())
            print(f"[SERVEUR] Message envoyé : {msg}")
            time.sleep(2)

# Démarrage du serveur dans un thread séparé (daemon pour arrêt automatique)
threading.Thread(target=start_socket_server, daemon=True).start()
time.sleep(2)  # Attendre que le serveur soit prêt

# PARTIE 3 : CRÉATION DU STREAMINGCONTEXT

# Le StreamingContext est le point d'entrée pour toute application Spark Streaming
# Il permet de définir la durée des micro-batches

from pyspark import SparkContext
from pyspark.streaming import StreamingContext

# Récupération ou création du SparkContext
sc = SparkContext.getOrCreate()
sc.setLogLevel("WARN")  # Réduire les logs pour plus de clarté

# Création du StreamingContext avec un batch interval de 5 secondes
ssc = StreamingContext(sc, 5)
print("[STREAMING] StreamingContext créé avec batch interval = 5 secondes")

# PARTIE 4 : LECTURE DES DONNÉES SOUS FORME DE DSTREAM

# Connexion au socket pour recevoir les données en flux continu
# Chaque ligne reçue sera traitée comme un élément du DStream

lines = ssc.socketTextStream("localhost", 9999)
print("[STREAMING] Connexion au socket établie")

# PARTIE 5 : APPLICATION DU WORDCOUNT STREAMING

# Transformation du flux pour compter les occurrences de chaque mot

# Étape 1 : Découper chaque ligne en mots (flatMap)
words = lines.flatMap(lambda line: line.split(" "))

# Étape 2 : Créer des paires (mot, 1) pour chaque mot
pairs = words.map(lambda w: (w, 1))

# Étape 3 : Agréger par clé (mot) en sommant les valeurs
counts = pairs.reduceByKey(lambda a, b: a + b)


# PARTIE 6 : AFFICHAGE DES RÉSULTATS DANS LA CONSOLE

# pprint() affiche les 10 premiers éléments de chaque RDD du DStream
counts.pprint()

# Démarrage du streaming
print("[STREAMING] Démarrage du traitement...")
ssc.start()

# Attendre 30 secondes puis arrêter
ssc.awaitTerminationOrTimeout(30)

# Arrêt propre du StreamingContext (sans arrêter le SparkContext)
ssc.stop(stopSparkContext=False)
print("[STREAMING] Arrêt du streaming")


[SERVEUR] En attente de connexion sur localhost:9999...
[STREAMING] StreamingContext créé avec batch interval = 5 secondes
[STREAMING] Connexion au socket établie
[STREAMING] Démarrage du traitement...
[SERVEUR] Client connecté depuis ('127.0.0.1', 53002)
[SERVEUR] Message envoyé : spark streaming dstream
-------------------------------------------
Time: 2026-01-16 20:14:20
-------------------------------------------
('streaming', 1)
('dstream', 1)
('spark', 1)

[SERVEUR] Message envoyé : spark spark streaming
[SERVEUR] Message envoyé : big data spark
[SERVEUR] Message envoyé : spark streaming dstream
-------------------------------------------
Time: 2026-01-16 20:14:25
-------------------------------------------
('streaming', 1)
('big', 1)
('spark', 3)
('data', 1)

[SERVEUR] Message envoyé : spark spark streaming
[SERVEUR] Message envoyé : big data spark
[SERVEUR] Message envoyé : spark streaming dstream
-------------------------------------------
Time: 2026-01-16 20:14:30
-----------

In [18]:
# VERSION MODIFIÉE - LECTURE DEPUIS UN FICHIER (CODE COMPLET)

import socket
import time
import threading
import random
from pyspark import SparkContext
from pyspark.streaming import StreamingContext



#  Fonction serveur socket qui lit depuis le fichier
def start_socket_server_from_file(filepath="messages.txt"):
    '''
    Serveur socket qui lit aléatoirement des lignes depuis un fichier
    et les envoie via socket.
    '''
    host = "localhost"
    port = 9999

    # Lecture du fichier
    with open(filepath, 'r', encoding='utf-8') as f:
        messages = [line.strip() for line in f.readlines() if line.strip()]

    if not messages:
        print("[ERREUR] Le fichier est vide")
        return

    print(f"[SERVEUR] {len(messages)} messages chargés depuis {filepath}")

    # Configuration du socket
    s = socket.socket()
    s.bind((host, port))
    s.listen(1)

    print(f"[SERVEUR] En attente de connexion sur {host}:{port}...")
    conn, addr = s.accept()
    print(f"[SERVEUR] Client connecté depuis {addr}")

    # Envoi aléatoire de lignes
    while True:
        msg = random.choice(messages)
        conn.send((msg + "\n").encode())
        print(f"[SERVEUR] Message envoyé : {msg}")
        time.sleep(2)

# Lancement du serveur dans un thread séparé
threading.Thread(target=start_socket_server_from_file, daemon=True).start()
time.sleep(2)  # Attendre que le serveur soit prêt

# Configuration de Spark Streaming
sc = SparkContext.getOrCreate()
sc.setLogLevel("WARN")

# Création du StreamingContext avec batch interval de 5 secondes
ssc = StreamingContext(sc, 5)
print("[STREAMING] StreamingContext créé avec batch interval = 5 secondes")

#  Connexion au socket et lecture des données
lines = ssc.socketTextStream("localhost", 9999)
print("[STREAMING] Connexion au socket établie")

#  Application du WordCount streaming
words = lines.flatMap(lambda line: line.split(" "))
pairs = words.map(lambda w: (w, 1))
counts = pairs.reduceByKey(lambda a, b: a + b)

#Affichage des résultats
counts.pprint()

# Démarrage et exécution du streaming
print("[STREAMING] Démarrage du traitement...")
ssc.start()

# Attendre 30 secondes puis arrêter
ssc.awaitTerminationOrTimeout(30)

# Arrêt propre du StreamingContext
ssc.stop(stopSparkContext=False)
print("[STREAMING] Arrêt du streaming")

Exception in thread Thread-19 (start_socket_server_from_file):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipython-input-1406584625.py", line 37, in start_socket_server_from_file
OSError: [Errno 98] Address already in use


[SERVEUR] 5 messages chargés depuis messages.txt
[STREAMING] StreamingContext créé avec batch interval = 5 secondes
[STREAMING] Connexion au socket établie
[STREAMING] Démarrage du traitement...
[SERVEUR] Client connecté depuis ('127.0.0.1', 51522)
[SERVEUR] Message envoyé : big data spark
[SERVEUR] Message envoyé : big data spark
[SERVEUR] Message envoyé : apache kafka spark
-------------------------------------------
Time: 2026-01-16 20:04:25
-------------------------------------------
('big', 2)
('data', 2)
('spark', 2)

[SERVEUR] Message envoyé : spark spark streaming
[SERVEUR] Message envoyé : real time processing
[SERVEUR] Message envoyé : spark streaming dstream
-------------------------------------------
Time: 2026-01-16 20:04:30
-------------------------------------------
('apache', 1)
('kafka', 1)
('streaming', 1)
('spark', 3)
('real', 1)
('time', 1)
('processing', 1)

[SERVEUR] Message envoyé : real time processing
[SERVEUR] Message envoyé : real time processing
------------

** RÉPONSES AUX QUESTIONS THÉORIQUES**


**QUESTION 4 : Qu'est-ce qu'un micro-batch dans DStream ?**

Un micro-batch est une petite unité de données collectées pendant un intervalle
de temps fixe (batch interval). Dans DStream :
- Les données du flux continu sont divisées en petits lots (batches)
- Chaque batch est traité comme un RDD indépendant
- La durée du batch détermine la latence du traitement
- C'est le principe fondamental du "Spark Streaming" : découper le flux
  continu en micro-batches pour les traiter avec l'API Spark classique

Exemple : avec un batch interval de 5 secondes, toutes les données reçues
pendant 5 secondes sont regroupées dans un RDD et traitées ensemble.


**QUESTION 5 : Sur quelle structure repose un DStream ?**

Un DStream (Discretized Stream) repose sur une séquence de RDDs :
- Chaque micro-batch correspond à un RDD
- Le DStream est une abstraction qui représente cette séquence continue de RDDs
- Les transformations sur un DStream (map, flatMap, reduce...) sont appliquées
  sur chaque RDD de la séquence
- Cela permet de réutiliser toute la puissance de Spark Core pour le streaming

Structure : DStream = Séquence de RDDs dans le temps
                      [RDD_t1, RDD_t2, RDD_t3, ...]


**QUESTION 6 : Quelle est la durée du batch utilisée ?**

Dans ce TP, la durée du batch est de 5 SECONDES.

Cette valeur est définie lors de la création du StreamingContext :
    ssc = StreamingContext(sc, 5)
                                ↑
                        Batch interval = 5 secondes

Cela signifie que :
- Toutes les 5 secondes, un nouveau RDD est créé avec les données reçues
- Le WordCount est calculé sur les données des 5 dernières secondes
- Les résultats sont affichés toutes les 5 secondes
